# Getting Started with clinops

`clinops` is a Clinical ML Pipeline Toolkit that handles the common plumbing every healthcare AI project needs:
loading MIMIC tables without hitting memory limits, clipping physiologically impossible values, normalizing
units across sites, building time-series windows with correct missingness handling, and splitting data without
leaking patients across folds.

This notebook walks through each module step by step using **synthetic data** — no MIMIC-IV access required.

## Contents
1. [Synthetic data setup](#1-synthetic-data-setup)
2. [Schema validation — `FlatFileLoader` + `ClinicalSchema`](#2-schema-validation)
3. [Outlier clipping — `ClinicalOutlierClipper`](#3-outlier-clipping)
4. [Unit normalization — `UnitNormalizer`](#4-unit-normalization)
5. [ICD code harmonization — `ICDMapper`](#5-icd-harmonization)
6. [Temporal windowing — `TemporalWindower`](#6-temporal-windowing)
7. [Imputation — `Imputer`](#7-imputation)
8. [Train/test splitting — `PatientSplitter`](#8-splitting)

---

In [ ]:
# Install clinops if not already installed
# !pip install clinops

import os
import tempfile
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

print("Imports OK")

## 1. Synthetic data setup

We simulate 8 ICU patients with hourly vital-sign measurements over 24 hours. The data mimics the
structure you would get from a real MIMIC-IV `chartevents` table — `subject_id`, `charttime`, and
a handful of measured vitals.

A few values are intentionally set outside physiological bounds (heart rate = 350, SpO2 = 20)
to demonstrate outlier clipping in the next step.

In [ ]:
rng = np.random.default_rng(0)
N_PATIENTS = 8
N_HOURS = 24
BASE_TIME = datetime(2150, 1, 1, 8, 0)   # MIMIC uses fictional future dates

rows = []
for pid in range(1, N_PATIENTS + 1):
    for h in range(N_HOURS):
        rows.append({
            "subject_id":  pid,
            "hadm_id":     1000 + pid,
            "charttime":   BASE_TIME + timedelta(hours=h),
            "heart_rate":  float(np.clip(rng.normal(75, 10), 50, 130)),
            "spo2":        float(np.clip(rng.normal(97,  2), 88, 100)),
            "resp_rate":   float(np.clip(rng.normal(16,  3),  8,  30)),
            "map":         float(np.clip(rng.normal(85, 12), 55, 130)),
        })

vitals = pd.DataFrame(rows)

# Inject a couple of impossible values so the clipper has something to do
vitals.loc[0, "heart_rate"] = 350.0   # impossible
vitals.loc[5, "spo2"]       =  20.0   # impossible

print(f"Shape: {vitals.shape}")
vitals.head()

## 2. Schema validation

`ClinicalSchema` lets you declare the expected structure of a DataFrame — required columns,
nullability constraints, and value bounds — before you do any processing.

`FlatFileLoader` applies that schema when it reads a CSV or Parquet file, raising
`SchemaValidationError` immediately if the data violates the contract rather than letting
corrupt values silently flow downstream.

**Why this matters in clinical ML:** A missing `subject_id` column or a glucose column
full of NaN values are bugs that often go unnoticed for hours. Schema validation catches
them at the boundary, where the error message is still meaningful.

In [ ]:
from clinops.ingest import ClinicalSchema, ColumnSpec, FlatFileLoader

schema = ClinicalSchema(
    name="vitals",
    columns=[
        ColumnSpec("subject_id",  nullable=False),
        ColumnSpec("heart_rate",  min_value=0,  max_value=300),
        ColumnSpec("spo2",        min_value=50, max_value=100),
        ColumnSpec("resp_rate",   min_value=0,  max_value=80),
        ColumnSpec("map",         min_value=20, max_value=250),
    ],
)

# Write synthetic data to a temp CSV and load it through the schema-aware loader
with tempfile.NamedTemporaryFile(suffix=".csv", mode="w", delete=False) as f:
    vitals.to_csv(f.name, index=False)
    tmp_csv = f.name

# strict=False → log violations instead of raising; strict=True would raise SchemaValidationError
loader = FlatFileLoader(tmp_csv, schema=schema, id_col="subject_id", strict=False)
validated_df = loader.load()

print(loader.summary())
os.unlink(tmp_csv)

## 3. Outlier clipping

`ClinicalOutlierClipper` uses *physiologically-grounded* bounds rather than statistical outlier
detection. A heart rate of 350 bpm is not a 3-sigma outlier — it is a data entry error or sensor
artifact. The pre-registered `VITAL_BOUNDS` and `LAB_BOUNDS` encode published physiological
limits for common measurements.

Three `action` modes:
- `"clip"` — replace out-of-range values with the bound (default, model-safe)
- `"null"` — replace with `NaN` (preserves the information that a value existed but was invalid)
- `"flag"` — add a boolean `{col}_outlier` column and leave the value unchanged

After transformation, `.report()` returns a per-column audit table.

In [ ]:
from clinops.preprocess import ClinicalOutlierClipper

clipper = ClinicalOutlierClipper(action="clip")
clipped = clipper.fit_transform(vitals)

# Inspect which columns had out-of-range values and how many were affected
report = clipper.report()
print("Outlier audit:")
print(report.to_string(index=False))

print(f"\nheart_rate before clipping: {vitals['heart_rate'].max():.0f} max")
print(f"heart_rate after  clipping: {clipped['heart_rate'].max():.0f} max")
print(f"spo2 before clipping: {vitals['spo2'].min():.0f} min")
print(f"spo2 after  clipping: {clipped['spo2'].min():.0f} min")

In [ ]:
# You can register custom bounds for columns not in VITAL_BOUNDS or LAB_BOUNDS
clipper_custom = ClinicalOutlierClipper(action="flag")
clipper_custom.add_bounds("resp_rate", low=4, high=60)  # tighter than default

flagged = clipper_custom.fit_transform(vitals)
# flagged now contains a resp_rate_outlier boolean column
if "resp_rate_outlier" in flagged.columns:
    print(f"Flagged {flagged['resp_rate_outlier'].sum()} resp_rate outliers")

## 4. Unit normalization

Multi-site EHR data routinely contains the same measurement in different units: glucose might
be `mg/dL` at one hospital and `mmol/L` at another. `UnitNormalizer` detects and converts
mixed-unit columns to a canonical target unit using a built-in conversion table (`UNIT_CONVERSIONS`)
with 30+ common clinical conversions.

After transformation, `.report()` shows which rows were converted and by what factor.

In [ ]:
from clinops.preprocess import UnitNormalizer

# Simulate glucose data from two sites with different units
glucose_df = pd.DataFrame({
    "subject_id":   list(range(1, 9)),
    "glucose":      [105.0, 98.0, 120.0, 87.0, 5.8, 7.2, 4.9, 6.5],
    "glucose_unit": ["mg/dL"] * 4 + ["mmol/L"] * 4,
})

print("Before normalization:")
print(glucose_df.to_string(index=False))

# column_unit_map tells the normalizer which column holds values and which holds units
normalizer = UnitNormalizer(column_unit_map={"glucose": "glucose_unit"})
normalized_df = normalizer.transform(glucose_df)

print("\nAfter normalization (all mg/dL):")
print(normalized_df.to_string(index=False))

print("\nConversions applied:")
print(normalizer.report().to_string(index=False))

## 5. ICD code harmonization

MIMIC-IV contains both ICD-9 (for historic admissions) and ICD-10 codes in the same
`diagnoses_icd` table. Harmonizing to a single coding system is necessary before you can
group diagnoses or use them as features.

`ICDMapper` ships with a curated mapping table and can also load the full CMS GEM
file (~72 k mappings) for production use. The `.chapter_series()` method maps any ICD-10
code to its chapter grouping (Circulatory, Endocrine, etc.), which is useful for
high-level comorbidity features.

In [ ]:
from clinops.preprocess import ICDMapper

# Mixed ICD-9 and ICD-10 diagnoses — exactly what MIMIC-IV looks like
dx_df = pd.DataFrame({
    "subject_id":   [1, 2, 3, 4, 5, 6],
    "icd_code":     ["I509",  "4280",  "E119",  "25000", "N179",  "5849"],
    "icd_version":  [10,       9,       10,       9,       10,       9],
})

print("Before harmonization:")
print(dx_df.to_string(index=False))

mapper = ICDMapper()
harmonized = mapper.harmonize(dx_df, code_col="icd_code", version_col="icd_version")
harmonized["chapter"] = mapper.chapter_series(harmonized["icd_code"])

print(f"\nAfter harmonization (mapper loaded {mapper.n_mappings:,} curated mappings):")
print(harmonized.to_string(index=False))

## 6. Temporal windowing

Longitudinal ICU data is most useful as a sequence of time-ordered feature windows.
`TemporalWindower` extracts sliding windows of a configurable size and stride, aggregating
measurements within each window into a single flat feature vector.

**Key parameters:**
- `window_hours` — length of each feature window
- `step_hours` — stride between successive windows (overlap = window - step)
- `min_observations` — drop sparse windows with fewer than N non-null measurements
- `imputation` — strategy applied within each window before aggregation

The output is a wide DataFrame: one row per patient per window, with `window_start`
and `window_end` columns alongside the aggregated features.

In [ ]:
from clinops.temporal import TemporalWindower, ImputationStrategy

windower = TemporalWindower(
    window_hours=6,       # 6-hour feature windows
    step_hours=3,         # new window every 3 hours (50% overlap)
    imputation=ImputationStrategy.FORWARD_FILL,
    min_observations=2,   # skip windows with < 2 non-null measurements
)

windows = windower.fit_transform(
    df=clipped,
    id_col="subject_id",
    time_col="charttime",
    feature_cols=["heart_rate", "spo2", "resp_rate", "map"],
)

print(f"Input:  {len(clipped):,} hourly measurements ({clipped['subject_id'].nunique()} patients)")
print(f"Output: {len(windows):,} feature windows")
print(f"Shape:  {windows.shape}")
print()
windows.head(6)

### Adding lag and rolling features

`LagFeatureBuilder` enriches the windowed DataFrame with lagged values and rolling
statistics. This lets a downstream model see not just the current window but recent
trends — useful for detecting deterioration.

Lags and rolling statistics are computed **per patient**, so patient 2's first window
never uses patient 1's last window as a lag.

In [ ]:
from clinops.temporal import LagFeatureBuilder

enriched = LagFeatureBuilder(
    lags=[1, 2],           # t-1 and t-2 windows
    rolling_windows=[4],   # 4-window rolling mean and std
    id_col="subject_id",
).fit_transform(windows)

new_cols = [c for c in enriched.columns if c not in windows.columns]
print(f"Added {len(new_cols)} lag/rolling features:")
for col in new_cols:
    print(f"  {col}")

print(f"\nFinal enriched shape: {enriched.shape}")

## 7. Imputation

Missing values are the norm in clinical data — not all vitals are measured every hour, labs
arrive sporadically, and some measurements are simply not ordered for certain patients.

`Imputer` must be fitted on the **training set only** and then applied to the test set.
This prevents any test-set statistics (means, medians) from leaking into the imputation
values used during training.

The `max_gap_hours` parameter prevents forward-fill from carrying a stale value across
a long gap where no measurement was taken — a common source of subtle errors in clinical
time-series work.

In [ ]:
from clinops.temporal import Imputer

# Temporal train/test split by window index (70/30)
split_idx = int(len(windows) * 0.7)
train_windows = windows.iloc[:split_idx].copy()
test_windows  = windows.iloc[split_idx:].copy()

# Inject some missingness into the test set
rng2 = np.random.default_rng(42)
missing_idx = rng2.choice(test_windows.index, size=max(1, len(test_windows) // 8), replace=False)
test_windows.loc[missing_idx, "heart_rate"] = np.nan

print(f"Train windows: {len(train_windows):,}")
print(f"Test  windows: {len(test_windows):,}")
print(f"Missing heart_rate in test: {test_windows['heart_rate'].isna().sum()}")

# Fit on train only — the training-set mean is used to fill test NaNs
imputer = Imputer(ImputationStrategy.MEAN)
imputer.fit(train_windows)
test_imputed = imputer.transform(test_windows)

print(f"Missing heart_rate after imputation: {test_imputed['heart_rate'].isna().sum()}")

## 8. Splitting

Standard random train/test splits are incorrect for clinical data because:
1. **Temporal leakage** — a model trained on data from 2155 and tested on 2150 is evaluated
   on its past, not its future.
2. **Patient leakage** — the same patient's earlier windows in train and later windows in test
   can make the model look better than it will generalize.

`clinops.split` provides three splitters that handle these correctly.

### PatientSplitter — no cross-patient leakage

In [ ]:
from clinops.split import PatientSplitter

result = PatientSplitter(
    id_col="subject_id",
    test_size=0.25,
    random_state=42,
).split(clipped)

print(result.summary())

train_pids = set(result.train["subject_id"].unique())
test_pids  = set(result.test["subject_id"].unique())
assert not (train_pids & test_pids), "Patient leakage detected!"
print(f"Train patients: {sorted(train_pids)}")
print(f"Test  patients: {sorted(test_pids)}")
print("No patient appears in both splits.")

### TemporalSplitter — strict time-based cut

All rows before the cutoff date go to train; all rows after go to test.
No information from the future is available during training.

In [ ]:
from clinops.split import TemporalSplitter

cutoff = BASE_TIME + timedelta(hours=int(N_HOURS * 0.75))
t_result = TemporalSplitter(
    cutoff=cutoff.isoformat(),
    time_col="charttime",
).split(clipped)

print(t_result.summary())
print(f"Cutoff: {cutoff}")
print(f"Latest train time: {t_result.train['charttime'].max()}")
print(f"Earliest test time: {t_result.test['charttime'].min()}")

### StratifiedPatientSplitter — preserves outcome rate

For imbalanced outcomes (e.g. ICU mortality rates of 10-20%), a random patient split
can put all the positive cases in one fold. `StratifiedPatientSplitter` ensures both
sets have the same positive rate while still respecting patient boundaries.

In [ ]:
from clinops.split import StratifiedPatientSplitter

# Attach a synthetic binary outcome — patients 1 and 2 are "non-survivors"
outcome_map = {pid: int(pid <= 2) for pid in range(1, N_PATIENTS + 1)}
clipped_with_outcome = clipped.copy()
clipped_with_outcome["hospital_expire_flag"] = clipped_with_outcome["subject_id"].map(outcome_map)

s_result = StratifiedPatientSplitter(
    id_col="subject_id",
    outcome_col="hospital_expire_flag",
    test_size=0.25,
    random_state=42,
).split(clipped_with_outcome)

print(s_result.summary())

# Confirm outcome rate is preserved
train_rate = s_result.train.drop_duplicates("subject_id")["hospital_expire_flag"].mean()
test_rate  = s_result.test.drop_duplicates("subject_id")["hospital_expire_flag"].mean()
print(f"Mortality rate — train: {train_rate:.2f}  test: {test_rate:.2f}")

---

## Summary

In this notebook we walked through all core `clinops` modules using synthetic data:

| Step | Module | Key class |
|------|--------|-----------|
| Schema validation | `clinops.ingest` | `FlatFileLoader`, `ClinicalSchema` |
| Outlier clipping | `clinops.preprocess` | `ClinicalOutlierClipper` |
| Unit normalization | `clinops.preprocess` | `UnitNormalizer` |
| ICD harmonization | `clinops.preprocess` | `ICDMapper` |
| Temporal windowing | `clinops.temporal` | `TemporalWindower` |
| Lag/rolling features | `clinops.temporal` | `LagFeatureBuilder` |
| Imputation | `clinops.temporal` | `Imputer` |
| Splitting | `clinops.split` | `PatientSplitter`, `TemporalSplitter`, `StratifiedPatientSplitter` |

## Next steps

- **Notebook 02** — end-to-end ICU mortality prediction pipeline with cohort alignment
- [Ingest guide](../../docs/guide/ingest.md) — MIMIC-IV, MIMIC-III, FHIR R4 loaders
- [Temporal guide](../../docs/guide/temporal.md) — windowing strategies and imputation tactics
- [Split guide](../../docs/guide/split.md) — choosing the right splitter for your study design
- [API Reference](../../docs/api/ingest.md) — full class and method signatures